In [ ]:
from datasets import load_dataset
import pandas as pd 

dataset = load_dataset("stanfordnlp/imdb")


In [ ]:
# använder hugging face dataset direkt utan att göra till df
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# tar bort train/test-dubbletter
train_text = set(train_dataset["text"])

test_dataset = test_dataset.filter(
    lambda example: example["text"] not in train_text
)

# delar train i train och val
split = train_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split["train"]
val_dataset = split["test"]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

# ladda tokenizer för DistilBERT
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_lenght=512)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# ladda ner klassificeringsmodellen. num_labels=2 för att recensionerna är antingen positiva eller negativa 
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2 
)

# ladda en data collator som gör padding (fyller ut meningarna till samma längd)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import Trainer, TrainingArguments
import numpy as np 
import evaluate 
import os

output_directory = os.path.abspath("./best_model")
os.makedirs(output_directory, exist_ok=True)

small_train_dataset = tokenized_train.shuffle(seed=42).select(range(5000))

# ladda mätvärden (ex accuracy)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir=output_directory,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)